# Proyek Akhir Praktikum Big Data
## TidarMart Data Platform: Analitik Big Data End to End pada Resource Terbatas

| | |
|:--|:--|
| **Nama** | Hilmi Mufid |
| **NPM** | 2505060046 |
| **Program Studi** | S1 Teknologi Informasi, Universitas Tidar |
| **SEED / VARIAN** | 60046 / 0 (fokus pembatalan dan logistik) |
| **Environment** | Ubuntu, conda env `bigdata`, Hadoop 3.4.3 (HDFS), PySpark 3.5.9, MongoDB 8.0 |
| **Isi notebook ini** | Tahap 1 (HDFS dan ingesti), Tahap 2 (profiling kualitas data), Tahap 5 (NoSQL MongoDB) |

**Cara menjalankan:** pastikan HDFS (`start-dfs.sh`) dan MongoDB (`sudo systemctl start mongod`) sudah aktif, buka notebook dari folder yang berisi `data_proyek/`, lalu pilih *Run All*. Seluruh sel berjalan dari atas ke bawah dan dapat diulang dari awal.

**Konfigurasi sumber daya:** Spark mode `local[2]`, `spark.driver.memory = 1536m`, `spark.sql.shuffle.partitions = 4`, adaptive query execution aktif. Tidak ada pemanggilan `toPandas()` atau `collect()` pada DataFrame besar, dan tidak ada `cache()`.

## Tahap 0. Persiapan SparkSession dan Fungsi Bantu

In [ ]:
import os
import sys
import glob
import json
import time
import subprocess
import itertools

import pandas as pd
import matplotlib.pyplot as plt

# Worker Python memakai interpreter yang sama dengan kernel Jupyter ini (agar pymongo dan lainnya tersedia)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.window import Window

# Konfigurasi Spark minimum yang diwajibkan soal (Bagian 4.2)
spark = (SparkSession.builder
         .appName("TidarMart")
         .master("local[2]")
         .config("spark.driver.memory", "1536m")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.adaptive.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Adaptive query execution:", spark.conf.get("spark.sql.adaptive.enabled"))

In [ ]:
# Identitas proyek (sesuai keluaran generator_data_proyek.py)
NPM, NAMA, SEED, VARIAN = "2505060046", "Hilmi Mufid", 60046, 0

# Alamat HDFS dan struktur direktori
HDFS = "hdfs://localhost:9000"
RAW_TRANSAKSI = "/tidarmart/raw/transaksi"
RAW_PELANGGAN = "/tidarmart/raw/pelanggan"
RAW_PRODUK = "/tidarmart/raw/produk"
RAW_STREAM_ARSIP = "/tidarmart/raw/stream_arsip"
STREAM_INPUT = "/tidarmart/stream_input"
CURATED = "/tidarmart/curated"
MODELS = "/tidarmart/models"

def jalankan(perintah, tampil=True, harus_sukses=True):
    """Menjalankan perintah terminal dan mengembalikan keluarannya sebagai teks."""
    hasil = subprocess.run(perintah, shell=True, capture_output=True, text=True)
    baris = [b for b in (hasil.stdout + hasil.stderr).splitlines() if "NativeCodeLoader" not in b]
    keluaran = "\n".join(baris).strip()
    if tampil and keluaran:
        print(keluaran)
    if harus_sukses and hasil.returncode != 0:
        raise RuntimeError(f"Perintah gagal (kode {hasil.returncode}): {perintah}")
    return hasil.stdout

print(f"Proyek TidarMart | {NAMA} | NPM {NPM} | SEED {SEED} | VARIAN {VARIAN}")

## Tahap 1. Penyimpanan Terdistribusi dan Ingesti (HDFS)

**Tujuan:** membangun struktur direktori di HDFS, mengunggah seluruh data multi format, membuktikan ukuran, jumlah blok, dan faktor replikasi, lalu memastikan tahap berikutnya membaca data dari HDFS (`hdfs://localhost:9000/...`), bukan dari path lokal.

**Karakteristik 5V yang dibuktikan:** Volume (jumlah baris dan ukuran berkas di HDFS) dan Variety (CSV, JSON Lines pelanggan, JSON produk).